# Aerobic Fermentation — Media Stoichiometry from CHNOSP Balance

Derives all stoichiometric coefficients from an elemental (CHNOSP) balance and
tracks how every medium component changes as glucose is consumed.

Two biomass elemental compositions are compared throughout:

| Name | Formula | Source |
|---|---|---|
| Roels extended | CH₁.₈O₀.₅N₀.₂S₀.₀₀₅P₀.₀₁ | Roels (1983), extended with S and P |
| Upcraft | CH₁.₉₈O₀.₆₄₇N₀.₁₇₃P₀.₀₁₅₁ | Upcraft's composition |

**Shared assumptions**
- Carbon + energy source: glucose (C₆H₁₂O₆)
- Nitrogen source: ammonium (NH₄⁺, oxidation state −3 — same as in biomass, so absent from the degree-of-reduction balance)
- Sulfur source: sulfate (SO₄²⁻) — relevant only for Roels extended
- Phosphorus source: dihydrogen phosphate (H₂PO₄⁻)
- Aerobic: O₂ is the terminal electron acceptor
- Yield of biomass on glucose: **Y_{X/S} = 0.4 g g⁻¹** (both compositions)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 1. Biomass compositions

Express biomass per **C-mol** (one formula unit per carbon atom):

$$\text{Biomass} = \text{CH}_{a}\text{O}_{b}\text{N}_{c}\text{S}_{d}\text{P}_{e}$$

The two compositions differ mainly in H/C, O/C and N/C; Upcraft's formula contains no sulfur.

In [ ]:
MW = dict(C=12.011, H=1.008, O=15.999, N=14.007, S=32.06, P=30.974)

# Elemental ratios per C-atom: (H/C, O/C, N/C, S/C, P/C)
COMPOSITIONS = {
    "Roels extended": (1.800, 0.500, 0.200, 0.005,  0.010),
    "Upcraft":        (1.980, 0.647, 0.173, 0.000,  0.0151),
}

def formula_str(a, b, c, d, e):
    s = f"CH{a}O{b}N{c}"
    if d: s += f"S{d}"
    s += f"P{e}"
    return s

rows = []
for name, (a, b, c, d, e) in COMPOSITIONS.items():
    MW_X    = MW['C'] + a*MW['H'] + b*MW['O'] + c*MW['N'] + d*MW['S'] + e*MW['P']
    gamma_X = 4 + a - 2*b - 3*c          # S/P contributions < 0.01 mol e⁻/C-mol each
    rows.append({"Name": name, "Formula": formula_str(a, b, c, d, e),
                 "MW (g/C-mol)": round(MW_X, 3), "γ (mol e⁻/C-mol)": round(gamma_X, 3)})

print(pd.DataFrame(rows).to_string(index=False))

## 2. Stoichiometric balance

Working per **gram of glucose consumed**.

**Carbon balance** (biomass + CO₂ = glucose carbon):
$$n_{CO_2} = n_{C,\text{glc}} - n_{C,X}$$

**Degree-of-reduction (electron) balance** (glucose electrons → biomass + O₂ reduction):
$$\gamma_S \, n_{C,\text{glc}} = \gamma_X \, n_{C,X} + 4 \, n_{O_2}$$

where $\gamma_S = 4$ mol e⁻/C-mol for glucose.

**N, S, P** — straight elemental balance from biomass composition.

**Hydrogen balance** — gives H₂O produced:
$$2\,n_{H_2O} = H_{\text{glc}} + 3\,n_{NH_3} - a\,n_{C,X}$$

In [ ]:
MW_glucose = 180.156;  Y_xs = 0.40;  gamma_S = 4.0

MW_O2  = 2*MW['O'];  MW_CO2 = MW['C'] + 2*MW['O']
MW_NH4 = MW['N'] + 4*MW['H'];  MW_SO4 = MW['S'] + 4*MW['O']
MW_PO4 = MW['P'] + 4*MW['O'] + 2*MW['H'];  MW_H2O = 2*MW['H'] + MW['O']

n_glc   = 1.0 / MW_glucose
n_C_glc = 6 * n_glc

def compute_stoich(a, b, c, d, e, Y_xs=Y_xs):
    """Return stoichiometric coefficients per gram of glucose consumed."""
    MW_X    = MW['C'] + a*MW['H'] + b*MW['O'] + c*MW['N'] + d*MW['S'] + e*MW['P']
    gamma_X = 4 + a - 2*b - 3*c
    n_C_X   = Y_xs / MW_X
    n_CO2   = n_C_glc - n_C_X
    n_O2    = (gamma_S * n_C_glc - gamma_X * n_C_X) / 4
    n_NH3   = c * n_C_X
    n_SO4   = d * n_C_X
    n_PO4   = e * n_C_X
    n_H2O   = (12*n_glc + 3*n_NH3 - a*n_C_X) / 2
    return dict(MW_X=MW_X, gamma_X=gamma_X, n_C_X=n_C_X,
                n_CO2=n_CO2, n_O2=n_O2, n_NH3=n_NH3,
                n_SO4=n_SO4, n_PO4=n_PO4, n_H2O=n_H2O,
                RQ=n_CO2/n_O2)

STOICH = {name: compute_stoich(*comps) for name, comps in COMPOSITIONS.items()}

# --- side-by-side stoichiometry table (g per g glucose) ---
SPECIES = ['Glucose', 'Biomass', 'O₂', 'CO₂', 'NH₄⁺', 'SO₄²⁻', 'H₂PO₄⁻', 'H₂O']
ROLES   = ['C+energy source', 'product', 'consumed', 'evolved',
           'N source', 'S source', 'P source', 'produced']

def g_per_g(st):
    return [-1.0, Y_xs,
            st['n_O2']*MW_O2,  st['n_CO2']*MW_CO2,
            st['n_NH3']*MW_NH4, st['n_SO4']*MW_SO4,
            st['n_PO4']*MW_PO4, st['n_H2O']*MW_H2O]

tbl = pd.DataFrame({'Species': SPECIES, 'Role': ROLES})
for name, st in STOICH.items():
    tbl[f"{name} (g/g_glc)"] = [f"{v:+.5f}" for v in g_per_g(st)]

print(tbl.to_string(index=False))
print()
for name, st in STOICH.items():
    print(f"  {name:20s}  RQ = {st['RQ']:.3f},  γ_X = {st['gamma_X']:.3f},  MW_X = {st['MW_X']:.3f} g/C-mol")

### Closure check — oxygen balance

In [ ]:
print("Oxygen balance closure per gram of glucose consumed:")
print()
for name, (a, b, c, d, e) in COMPOSITIONS.items():
    st = STOICH[name]
    O_in  = 6*n_glc + 2*st['n_O2']
    O_out = b*st['n_C_X'] + 2*st['n_CO2'] + st['n_H2O']
    print(f"  {name:20s}  O_in = {O_in:.6f},  O_out = {O_out:.6f},  "
          f"imbalance = {abs(O_in-O_out):.2e}  (mol-O / g_glc)")

## 3. Media composition vs. extent of glucose consumption

Define an initial medium, then sweep glucose consumed from 0 → S₀.

In [ ]:
S0_glucose = 20.0   # g/L initial glucose
X_0        = 0.05   # g/L inoculum biomass

# Initial medium per composition: 20% excess over stoichiometric demand
MEDIUM = {}
for name, st in STOICH.items():
    MEDIUM[name] = dict(
        NH4_0 = 1.2 * st['n_NH3'] * S0_glucose,   # mol/L
        PO4_0 = 1.2 * st['n_PO4'] * S0_glucose,   # mol/L
        SO4_0 = 1.2 * st['n_SO4'] * S0_glucose,   # mol/L
    )

print(f"{'Component':<20}  {'Roels extended':>18}  {'Upcraft':>12}")
print(f"{'Glucose (g/L)':<20}  {S0_glucose:>18.1f}  {S0_glucose:>12.1f}")
for key, label, mw in [('NH4_0', 'NH₄⁺ (mmol/L)', MW_NH4),
                        ('PO4_0', 'H₂PO₄⁻ (mmol/L)', MW_PO4),
                        ('SO4_0', 'SO₄²⁻ (mmol/L)', MW_SO4)]:
    r = MEDIUM["Roels extended"][key] * 1e3
    u = MEDIUM["Upcraft"][key] * 1e3
    print(f"{label:<20}  {r:>18.2f}  {u:>12.2f}")

In [ ]:
dS = np.linspace(0, S0_glucose, 300)   # g/L glucose consumed

# Pre-compute all sweep arrays, keyed by composition name
SWEEPS = {}
for name, st in STOICH.items():
    med = MEDIUM[name]
    SWEEPS[name] = dict(
        glucose = S0_glucose - dS,
        biomass = X_0 + Y_xs * dS,
        O2_dem  = st['n_O2']  * dS * MW_O2,          # g/L cumulative
        CO2_ev  = st['n_CO2'] * dS * 1e3,             # mmol/L cumulative
        NH4     = (med['NH4_0'] - st['n_NH3']*dS)*1e3,  # mmol/L
        PO4     = (med['PO4_0'] - st['n_PO4']*dS)*1e3,  # mmol/L
        SO4     = (med['SO4_0'] - st['n_SO4']*dS)*1e3,  # mmol/L
    )

print("Sweeps computed for:", list(SWEEPS))

## 4. Plots

In [ ]:
COLORS = {"Roels extended": "C0", "Upcraft": "C1"}
LSTYLE = {"Roels extended": "-",  "Upcraft": "--"}

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.suptitle(
    f"Aerobic growth on glucose  |  $Y_{{X/S}}$ = {Y_xs} g/g  |  "
    "blue = Roels extended,  orange = Upcraft",
    fontsize=11,
)

panels = [
    ("glucose", "Glucose",              "g / L"),
    ("biomass", "Biomass",              "g / L"),
    ("O2_dem",  "Cumulative O₂ demand", "g / L"),
    ("CO2_ev",  "Cumulative CO₂",       "mmol / L"),
    ("NH4",     "NH₄⁺",                 "mmol / L"),
    ("PO4",     "H₂PO₄⁻",              "mmol / L"),
]

for ax, (key, title, ylabel) in zip(axes.flat, panels):
    for name, sw in SWEEPS.items():
        ax.plot(dS, sw[key], color=COLORS[name], ls=LSTYLE[name], label=name)
    ax.set_title(title);  ax.set_ylabel(ylabel)
    ax.set_xlabel("Glucose consumed  (g / L)");  ax.grid(alpha=0.3)

axes.flat[2].legend(fontsize=8)   # legend on O2 panel (lines are most separated there)
plt.tight_layout()
plt.show()

## 5. Summary table at full glucose consumption

In [ ]:
rows = []
for name, st in STOICH.items():
    med = MEDIUM[name]
    rows.append({
        "Composition"      : name,
        "Glucose initial"  : f"{S0_glucose:.1f}",
        "Glucose final"    : "0",
        "Biomass initial"  : f"{X_0:.2f}",
        "Biomass final"    : f"{X_0 + Y_xs*S0_glucose:.2f}",
        "O₂ demand (g/L)"  : f"{st['n_O2']*S0_glucose*MW_O2:.2f}",
        "CO₂ evolved (g/L)": f"{st['n_CO2']*S0_glucose*MW_CO2:.2f}",
        "NH₄⁺ init (g/L)"  : f"{med['NH4_0']*MW_NH4:.2f}",
        "NH₄⁺ final (g/L)" : f"{(med['NH4_0']-st['n_NH3']*S0_glucose)*MW_NH4:.2f}",
        "H₂PO₄⁻ init (g/L)": f"{med['PO4_0']*MW_PO4:.3f}",
        "H₂PO₄⁻ final(g/L)": f"{(med['PO4_0']-st['n_PO4']*S0_glucose)*MW_PO4:.3f}",
        "RQ"               : f"{st['RQ']:.3f}",
    })

summary = pd.DataFrame(rows).T
summary.columns = list(STOICH)
print(summary.to_string())

## 6. Dynamic batch simulation — PyOMES model architecture

The stoichiometric analysis above gives endpoint compositions from elemental balance alone.
This section runs the same fermentation **dynamically** through the PyOMES batch fermentation
framework, which resolves:

- Monod-limited growth kinetics on glucose
- kLa-based O₂/CO₂ gas–liquid transfer
- Full acid–base equilibria: carbonate (pKₐ 6.35 / 10.33), phosphate (pKₐ 2.15 / 7.20 / 12.35),
  bisulfate (pKₐ 1.99), and ammonium (pKₐ 9.25)

**Initial medium** (same for both compositions):

| Component | g / L | Dominant ion(s) |
|---|---|---|
| Glucose | 30.0 | — |
| KH₂PO₄ | 30.0 | H₂PO₄⁻, K⁺ |
| NH₄Cl | 5.0 | NH₄⁺, Cl⁻ |
| K₂SO₄ | 0.5 | K⁺, SO₄²⁻ |
| MgSO₄·7H₂O | 0.4 | Mg²⁺, SO₄²⁻ |

In [ ]:
import sys
from pathlib import Path

# Bootstrap: notebook is in demos/; _bootstrap.py is in the same directory.
_here = Path().resolve()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))
import _bootstrap  # noqa: F401 — adds models/ to sys.path

from vlmodels.fermenter.config import FermenterBuilder
from PyOMES.reactions import ReactionSystem, ReactionBuilder
from PyOMES.chemistry.databases.bioprocess_basic import BIOPROCESS_BASIC

In [ ]:
# ── Vessel and process parameters ─────────────────────────────────────
V_TOTAL_L   = 5.0       # L total fermenter volume
HEADSPACE_F = 0.20      # fractional headspace
T_K         = 310.15    # 37 °C
MU_MAX_H    = 0.50      # h⁻¹  maximum specific growth rate
KS_G_L      = 0.05      # g / L  glucose half-saturation (Ks)
KLA_O2      = 200.0     # h⁻¹  volumetric O₂ mass-transfer coefficient
KLA_CO2_R   = 0.90      # kLa(CO₂) / kLa(O₂) ratio
X0_G_L      = 0.05      # g / L  inoculum biomass
Y_XS        = 0.40      # g biomass / g glucose

# ── Component molecular weights for medium preparation ─────────────────
_MW_MED = dict(
    Glucose=180.156, KH2PO4=136.086, NH4Cl=53.491,
    K2SO4=174.259,   MgSO4_7H2O=246.474,
)
_V_LIQ = V_TOTAL_L * (1.0 - HEADSPACE_F)

# ── Import Species objects that must be shared with equilibrium reactions
# BIOPROCESS_BASIC equilibria use common_species objects directly, so the
# kinetic reaction stoichiometry must reference the exact same instances
# to pass check_species_consistency (which raises on charge/MW conflicts).
from PyOMES.chemistry.common_species import CO2 as _CO2_sp, H2O as _H2O_sp, NH4_plus as _NH4_sp
from PyOMES.chemistry.species import Species
from PyOMES.reactions.stoichiometry import StoichiometryEntry
from PyOMES.reactions.kinetic import KineticReaction
from PyOMES.core.solvers import SimultaneousAdaptiveSolver


def _seed_medium(liq, biomass_id, MW_X):
    """Inject initial medium species into a LiquidPhase after the builder creates the CV."""
    V  = liq.V_L
    nm = liq.n_mol

    nm["Glucose"]  = (30.0 / _MW_MED["Glucose"]) * V

    # KH₂PO₄ → H₂PO₄⁻ + K⁺
    n_kp = (30.0 / _MW_MED["KH2PO4"]) * V
    nm["H2PO4-"]  = n_kp
    nm["K+"]      = nm.get("K+", 0.0) + n_kp

    # NH₄Cl → NH₄⁺ + Cl⁻
    n_ac = (5.0  / _MW_MED["NH4Cl"])   * V
    nm["NH4+"]    = n_ac
    nm["Cl-"]     = n_ac

    # K₂SO₄ → 2 K⁺ + SO₄²⁻
    n_ks = (0.5  / _MW_MED["K2SO4"])   * V
    nm["SO4--"]   = nm.get("SO4--", 0.0) + n_ks
    nm["K+"]      = nm.get("K+", 0.0)   + 2.0 * n_ks

    # MgSO₄·7H₂O → Mg²⁺ + SO₄²⁻
    n_mg = (0.4  / _MW_MED["MgSO4_7H2O"]) * V
    nm["SO4--"]   = nm.get("SO4--", 0.0) + n_mg
    nm["Mg++"]    = n_mg

    # Inoculum and H⁺ seed (overwritten by speciation on first solve)
    nm[biomass_id] = (X0_G_L / MW_X) * V
    nm["H+"]       = 1e-7 * V


def build_sim(comp_name, a, b, c, d, e):
    """Build, seed, and return a Simulation for the given biomass elemental ratios.

    The kinetic reaction is built manually so that the NH4+, CO2, and H2O
    Species objects are identical to those in the BIOPROCESS_BASIC equilibria
    (required by check_species_consistency, which raises on charge/MW conflicts).
    The reaction system combines simple Monod aerobic growth (NH4+ as N source)
    with all BIOPROCESS_BASIC equilibria (carbonate, phosphate, bisulfate, ammonium).
    The simulation uses an adaptive DOP853 solver so step sizes automatically
    contract when dissolved O₂ changes rapidly — no O₂ co-limitation term needed.
    """
    MW_X   = MW["C"] + a*MW["H"] + b*MW["O"] + c*MW["N"] + d*MW["S"] + e*MW["P"]
    bm_id  = "Biomass_" + comp_name.replace(" ", "_")
    bm_atoms = {"C": 1.0, "H": float(a), "O": float(b), "N": float(c)}
    if d: bm_atoms["S"] = float(d)
    if e: bm_atoms["P"] = float(e)

    # ── Simple Monod rate: mol glucose consumed h⁻¹ (extensive) ─────────
    def rate_fn(env, _bi=bm_id, _mwx=MW_X, _mws=_MW_MED["Glucose"]):
        C_S  = env.concentrations.get("Glucose", 0.0)   # mol / L
        C_X  = env.concentrations.get(_bi,       0.0)   # mol / L
        S_gL = C_S * _mws
        X_gL = C_X * _mwx
        if X_gL <= 1e-30 or S_gL <= 0.0:
            return 0.0
        mu = MU_MAX_H * S_gL / (KS_G_L + S_gL)
        return (mu / Y_XS) * X_gL / _mws * env.V_L

    # ── CHNO stoichiometry per mol glucose (NH₄⁺ as N source) ───────────
    # Must use common_species objects for NH4+, CO2, H2O to avoid the
    # SpeciesConflictError raised by check_species_consistency when the
    # same species ID appears with different MW or charge across reactions.
    Y_mol = Y_XS * _MW_MED["Glucose"] / MW_X
    nCO2  = 6.0 - Y_mol * 1.0
    nNH4  = Y_mol * float(c)
    nH2O  = (12.0 + 4.0 * nNH4 - Y_mol * float(a)) / 2.0
    nO2   = (Y_mol * float(b) + 2.0 * nCO2 + nH2O - 6.0) / 2.0

    _O2_sp   = Species(id="O2",      atoms={"O": 2},            charge=0, MW=31.998)
    _GLUC_sp = Species(id="Glucose", atoms={"C":6,"H":12,"O":6},charge=0, MW=_MW_MED["Glucose"])
    _BM_sp   = Species(id=bm_id,     atoms=bm_atoms,            charge=0, MW=MW_X)

    kinetic_rxn = KineticReaction(
        stoichiometry=[
            StoichiometryEntry(_GLUC_sp, "liquid", -1.0   ),
            StoichiometryEntry(_O2_sp,   "liquid", -nO2   ),
            StoichiometryEntry(_NH4_sp,  "liquid", -nNH4  ),
            StoichiometryEntry(_BM_sp,   "liquid",  Y_mol ),
            StoichiometryEntry(_CO2_sp,  "liquid",  nCO2  ),
            StoichiometryEntry(_H2O_sp,  "liquid",  nH2O  ),
        ],
        rate_fn=rate_fn,
        balance_elements=["C", "H", "N", "O"],
        label=f"growth_{bm_id}",
    )

    # ── Combined: kinetic + full BIOPROCESS_BASIC equilibria ────────────
    rxn_sys = ReactionSystem(
        [kinetic_rxn] + list(BIOPROCESS_BASIC.reactions),
        label=f"aerobic_ferm_{bm_id}",
    )

    sim = (
        FermenterBuilder()
        .vessel(V_total_L=V_TOTAL_L, headspace_frac=HEADSPACE_F, T_K=T_K)
        .gas_feed(vvm_min=1.0, composition={"O2": 0.21, "N2": 0.79})
        .transfer_kinetic(kLa_O2=KLA_O2, kLa_CO2_ratio=KLA_CO2_R)
        .chemistry()
        .reaction_system(rxn_sys)
        .build_simulation()
    )

    # Adaptive DOP853 solver — step size contracts automatically under rapid
    # O₂ drawdown at peak growth, maintaining tolerances without a fixed step.
    # n_steps in sim.run() becomes the output grid resolution, not solver steps.
    sim.solver = SimultaneousAdaptiveSolver(rtol=1e-4, atol=1e-9, max_step=0.1)

    _seed_medium(sim.cvs["main"].phases["liquid"], bm_id, MW_X)
    return sim, bm_id, MW_X


print("Helpers defined — ready to build simulations.")

In [ ]:
TAU_H   = 60.0   # h — long enough to exhaust 30 g/L glucose
N_STEPS = 300    # output grid resolution (adaptive solver chooses its own internal steps)

results  = {}
bm_ids   = {}
MW_Xs    = {}

for comp_name, (a, b, c, d, e) in COMPOSITIONS.items():
    print(f"Building and running: {comp_name} ...", end="", flush=True)
    sim, bm_id, MW_X = build_sim(comp_name, a, b, c, d, e)
    bm_ids[comp_name]  = bm_id
    MW_Xs[comp_name]   = MW_X

    result = sim.run(tau_h=TAU_H, n_steps=N_STEPS)
    results[comp_name] = result

    lm     = result.liquid_mol["main"]
    pH     = result.pH["main"]
    n_g0   = lm.get("Glucose", [0.0])[0]
    n_gf   = lm.get("Glucose", [0.0])[-1]
    conv   = (n_g0 - n_gf) / n_g0 * 100.0 if n_g0 > 0 else 0.0
    print(f" done.  conversion = {conv:.1f}%   pH: {pH[0]:.2f} -> {pH[-1]:.2f}")

In [ ]:
def print_conc_report(result, comp_name, bm_id, MW_X):
    """Print initial and final liquid-phase concentrations and pH."""
    lm  = result.liquid_mol["main"]
    pH  = result.pH["main"]
    V   = _V_LIQ

    n_g0  = lm.get("Glucose", [0.0])[0]
    n_gf  = lm.get("Glucose", [0.0])[-1]
    conv  = (n_g0 - n_gf) / n_g0 * 100.0 if n_g0 > 0 else 0.0

    print(f"\n{'='*72}")
    print(f"  {comp_name}   (MW_X = {MW_X:.3f} g C-mol⁻¹)")
    print(f"  Initial pH = {pH[0]:.2f}   |   Final pH = {pH[-1]:.2f}")
    print(f"  Glucose conversion = {conv:.1f}%")
    print(f"{'='*72}")
    print(f"  {'Species':<16}  {'Initial':>12}  {'Final':>12}  {'Change':>12}")
    print(f"  {'-'*56}")

    # Biomass (g / L)
    MW_Xk = MW_X
    for n0, nf, label, mw, unit in [
        (lm.get(bm_id, [0.0])[0], lm.get(bm_id, [0.0])[-1], "Biomass",    MW_Xk,   "g/L"),
        (n_g0,                     n_gf,                       "Glucose",    _MW_MED["Glucose"], "g/L"),
    ]:
        c0 = n0 / V * mw;  cf = nf / V * mw
        print(f"  {label:<16}  {c0:>10.3f} {unit}  {cf:>10.3f} {unit}  {cf-c0:>+10.3f} {unit}")

    # Ionic / dissolved species (mmol / L)
    SPECIES_ROWS = [
        ("NH4+",    "NH4+",     18.038),
        ("NH3",     "NH3",      17.031),
        ("H2PO4-",  "H2PO4-",   96.986),
        ("HPO4--",  "HPO4--",   95.978),
        ("SO4--",   "SO4--",    96.062),
        ("K+",      "K+",       39.098),
        ("Cl-",     "Cl-",      35.453),
        ("Mg++",    "Mg++",     24.305),
        ("CO2",     "CO2(aq)",  44.010),
        ("HCO3-",   "HCO3-",    61.016),
        ("H+",      "H+",        1.008),
        ("OH-",     "OH-",      17.008),
        ("O2",      "O2(aq)",   32.000),
    ]
    for sp_id, label, _ in SPECIES_ROWS:
        n0 = lm.get(sp_id, [0.0])[0];  nf = lm.get(sp_id, [0.0])[-1]
        c0 = n0 / V * 1e3;  cf = nf / V * 1e3    # mmol / L
        print(f"  {label:<16}  {c0:>10.4f} mM   {cf:>10.4f} mM   {cf-c0:>+10.4f} mM")

    # Total phosphate check
    sp_P = ["H3PO4", "H2PO4-", "HPO4--", "PO4---"]
    P0   = sum(lm.get(s, [0.0])[0]  for s in sp_P) / V * 1e3
    Pf   = sum(lm.get(s, [0.0])[-1] for s in sp_P) / V * 1e3
    print(f"  {'CT_P (sum)':<16}  {P0:>10.4f} mM   {Pf:>10.4f} mM   {Pf-P0:>+10.4f} mM")


for comp_name in COMPOSITIONS:
    print_conc_report(
        results[comp_name], comp_name,
        bm_ids[comp_name], MW_Xs[comp_name],
    )

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(
    f"Aerobic batch fermentation — PyOMES dynamic simulation\n"
    f"V = {V_TOTAL_L} L,  T = {T_K-273.15:.0f} °C,  µ_max = {MU_MAX_H} h⁻¹,  "
    f"kLa(O₂) = {KLA_O2:.0f} h⁻¹",
    fontsize=10,
)

panels = [
    ("Glucose",  "Glucose",     "g / L",     _MW_MED["Glucose"]),
    (None,       "Biomass",     "g / L",     None              ),
    ("NH4+",     "NH4+",        "mmol / L",  18.038            ),
    ("H2PO4-",   "H2PO4-",      "mmol / L",  96.986            ),
    ("O2",       "O₂(aq)",      "mmol / L",  32.000            ),
    (None,       "pH",          "",          None              ),
]

for ax, (sp_id, title, ylabel, mw_sp) in zip(axes.flat, panels):
    for name, result in results.items():
        t  = result.t_h
        lm = result.liquid_mol["main"]

        if title == "Biomass":
            bm_id = bm_ids[name];  mw = MW_Xs[name]
            y = np.array(lm.get(bm_id, [0.0]*len(t))) / _V_LIQ * mw
        elif title == "pH":
            y = result.pH["main"]
        elif ylabel == "g / L":
            y = np.array(lm.get(sp_id, [0.0]*len(t))) / _V_LIQ * mw_sp
        else:
            y = np.array(lm.get(sp_id, [0.0]*len(t))) / _V_LIQ * 1e3

        ax.plot(t, y, color=COLORS[name], ls=LSTYLE[name], label=name)

    ax.set_title(title);  ax.set_ylabel(ylabel);  ax.set_xlabel("Time (h)")
    ax.grid(alpha=0.3)

axes.flat[0].legend(fontsize=8)
plt.tight_layout()

_out = Path("aerobic_fermentation/simulation_timeseries.png")
plt.savefig(_out, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {_out}")